# Subagent

In [1]:
# pip install -U langchain langgraph langchain-openai langchain-community duckduckgo-search

In [2]:
from dataclasses import dataclass
from typing import Any

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langchain_community.tools import DuckDuckGoSearchResults

In [ ]:
# -----------------------------------
# 1. 사용자 context
# -----------------------------------
@dataclass
class UserContext:
    user_id: str


# -----------------------------------
# 2. 장기 메모리 저장소
# -----------------------------------
store = InMemoryStore()


# -----------------------------------
# 3. DuckDuckGo 검색 도구 준비
# -----------------------------------
# output_format="list" 로 받으면 검색 결과를 구조화된 형태로 다루기 쉽습니다.
ddg_tool = DuckDuckGoSearchResults(
    output_format="list",
    max_results=8,
)


# -----------------------------------
# 4. 좋아하는 영화 장르 저장 tool
# -----------------------------------
@tool
def save_favorite_genre(genre: str, runtime: ToolRuntime[UserContext]) -> str:
    """
    사용자가 좋아하는 영화 장르를 장기 메모리에 저장합니다.
    예: genre='sci-fi'
    """
    assert runtime.store is not None

    namespace = ("users", runtime.context.user_id, "movie_preferences")

    runtime.store.put(
        namespace,
        "favorite_genre",
        {"genre": genre}
    )

    return f"좋아하는 영화 장르를 '{genre}'로 저장했습니다."


# -----------------------------------
# 5. 저장된 장르 조회 tool
# -----------------------------------
@tool
def load_favorite_genre(runtime: ToolRuntime[UserContext]) -> str:
    """
    저장된 영화 장르를 조회합니다.
    """
    assert runtime.store is not None

    namespace = ("users", runtime.context.user_id, "movie_preferences")
    memory = runtime.store.get(namespace, "favorite_genre")

    if memory is None:
        return "아직 저장된 좋아하는 영화 장르가 없습니다."

    return f"저장된 좋아하는 영화 장르는 '{memory.value['genre']}' 입니다."


# -----------------------------------
# 6. 최근 영화 검색 + 추천 전용 서브에이전트
# -----------------------------------
movie_recommender_agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[],
    system_prompt=(
        "당신은 영화 추천 전문가입니다. "
        "입력으로 사용자의 선호 장르와 최신 웹 검색 결과가 주어집니다. "
        "검색 결과에 나온 최근 영화들 중에서 그 장르에 맞는 작품 3~5편을 골라 추천하세요. "
        "반드시 아래 원칙을 따르세요:\n"
        "1. 검색 결과에 근거해 추천하세요.\n"
        "2. 검색 결과에 없는 영화를 지어내지 마세요.\n"
        "3. 각 영화마다 추천 이유를 1문장씩 쓰세요.\n"
        "4. 마지막에 '왜 이 영화를 골랐는지'를 짧게 요약하세요.\n"
        "5. 답변은 한국어로 하세요."
    ),
)


# -----------------------------------
# 7. 검색 기반 추천 tool
# -----------------------------------
@tool
def recommend_recent_movies(runtime: ToolRuntime[UserContext]) -> str:
    """
    저장된 좋아하는 장르를 바탕으로 DuckDuckGo에서 최근 영화를 검색한 뒤 추천합니다.
    """
    assert runtime.store is not None

    namespace = ("users", runtime.context.user_id, "movie_preferences")
    memory = runtime.store.get(namespace, "favorite_genre")

    if memory is None:
        return "좋아하는 영화 장르가 저장되어 있지 않습니다. 먼저 좋아하는 장르를 알려주세요."

    genre = memory.value["genre"]

    # 최근작 중심으로 검색
    search_query = (
        f"넷플릭스영화중 {genre}장르의 영화를 찾아줘. 영화 이름하고 장르로 데이터를 가져와줘. "
    )

    try:
        search_results = ddg_tool.invoke(search_query)
    except Exception as e:
        return f"영화 검색 중 오류가 발생했습니다: {e}"

    if not search_results:
        return f"'{genre}' 장르에 대한 최근 영화 검색 결과를 찾지 못했습니다."

    
    print('end==search_results==',search_results)
    result = movie_recommender_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"사용자가 좋아하는 영화 장르는 '{genre}' 입니다.\n\n"
                        f"아래는 DuckDuckGo로 찾은 최근 영화 관련 검색 결과입니다.\n\n"
                        f"{search_results}\n\n"
                        f"이 자료를 바탕으로 추천해주세요."
                    ),
                }
            ]
        }
    )

    return result["messages"][-1].content


# -----------------------------------
# 8. 메인 에이전트
# -----------------------------------
agent = create_agent(
    model="gpt-5-mini",
    tools=[save_favorite_genre, load_favorite_genre, recommend_recent_movies],
    store=store,
    context_schema=UserContext,
    system_prompt=(
        "당신은 사용자의 영화 취향을 기억하고 추천해주는 도우미입니다.\n"
        "- 사용자가 좋아하는 영화 장르를 말하면 save_favorite_genre를 사용하세요.\n"
        "- 사용자가 저장된 장르를 물어보면 load_favorite_genre를 사용하세요.\n"
        "- 사용자가 영화 추천을 요청하면 recommend_recent_movies를 사용하세요.\n"
        "- 답변은 한국어로 하세요."
    ),
)


# -----------------------------------
# 9. 실행 예시
# -----------------------------------
if __name__ == "__main__":
    user_context = UserContext(user_id="user_001")

    # 1) 장르 저장
    response1 = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": "나는 영화 액션 장르를 좋아해. 기억해줘."}
            ]
        },
        context=user_context,
    )
    print("=== 응답 1 ===")
    print(response1["messages"][-1].content)

    # 2) 장르 확인
    response2 = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": "내가 좋아하는 영화 장르가 뭐였지?"}
            ]
        },
        context=user_context,
    )
    print("\n=== 응답 2 ===")
    print(response2["messages"][-1].content)

    # 3) 최근 영화 추천
    response3 = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": "내 취향에 맞는 최근 영화 추천해줘."}
            ]
        },
        context=user_context,
    )
    print("\n=== 응답 3 ===")
    print(response3["messages"][-1].content)